In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import theilslopes

def robust_sigma(residuals):
    med = np.median(residuals)
    mad = np.median(np.abs(residuals - med))
    return 1.4826 * mad

def compute_metrics_old(df, BIN_WIDTH=50, MIN_BINS=5):
    
    # ---------------------------------
    # FILTER: classification == 4 only
    # ---------------------------------
    df = df[df["classification"] == 4].copy()
    
    if df.empty:
        return None

    # ---------------------------------
    # Compute WSE
    # ---------------------------------
    df["wse"] = df["height"] - df["geoid"]
    df = df[np.isfinite(df["wse"])]
    
    if df.empty:
        return None

    # ---------------------------------
    # Bin by distance
    # ---------------------------------
    df["bin"] = (df["d_m"] // BIN_WIDTH).astype(int)
    grouped = df.groupby("bin")

    bin_centers = []
    medians = []

    for b, g in grouped:
        bin_centers.append((b + 0.5) * BIN_WIDTH)
        medians.append(np.median(g["wse"]))

    if len(bin_centers) < MIN_BINS:
        return None

    d_vals = np.array(bin_centers)
    wse_vals = np.array(medians)

    # ---------------------------------
    # Theil-Sen slope
    # ---------------------------------
    slope, intercept, _, _ = theilslopes(wse_vals, d_vals)

    residuals = wse_vals - (intercept + slope * d_vals)
    sigma = robust_sigma(residuals)

    canal_length = d_vals.max() - d_vals.min()

    # ---------------------------------
    # Coverage + Continuity
    # ---------------------------------
    bins_sorted = np.sort(df["bin"].unique())

    min_bin = bins_sorted.min()
    max_bin = bins_sorted.max()
    total_possible_bins = max_bin - min_bin + 1

    coverage_frac = len(bins_sorted) / total_possible_bins

    # Longest contiguous run
    max_run = 1
    current_run = 1

    for i in range(1, len(bins_sorted)):
        if bins_sorted[i] == bins_sorted[i - 1] + 1:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1

    contig_frac = max_run / total_possible_bins

    # ---------------------------------
    # Signal-to-noise
    # ---------------------------------
    abs_slope = abs(slope)
    signal_amp = abs_slope * canal_length

    slope_snr = signal_amp / sigma if sigma > 0 else np.nan

    return {
        "n_bins": len(d_vals),
        "slope_m_per_m": slope,
        "abs_slope_m_per_m": abs_slope,
        "residual_sigma_robust_m": sigma,
        "canal_length_m": canal_length,
        "coverage_frac": coverage_frac,
        "contig_frac": contig_frac,
        "slope_snr": slope_snr,
    }

from pathlib import Path
import pandas as pd

input_dir = Path("/water1/mriduls/SOGRAIN_Analysis/us-south_from_204_to_25711/extracted_points")
output_dir = Path("/water1/mriduls/SOGRAIN_Analysis/us-south_from_204_to_25711/wse_metrics_old")
output_dir.mkdir(exist_ok=True)

records = []

# ------------------------------------------------------------
# Loop over chunk folders only
# ------------------------------------------------------------
chunk_dirs = sorted([d for d in input_dir.glob("chunk_*") if d.is_dir()])

print(f"Found {len(chunk_dirs)} chunk folders")

for chunk_dir in chunk_dirs:

    flag_path = chunk_dir / "extraction_complete.flag"

    # Skip unfinished chunks
    if not flag_path.exists():
        print(f"Skipping {chunk_dir.name} (still running)")
        continue

    print(f"Processing {chunk_dir.name}")

    parquet_files = sorted(chunk_dir.glob("*.parquet"))

    for parquet_file in parquet_files:
        grain_id = parquet_file.stem

        try:
            df = pd.read_parquet(
                parquet_file,
                columns=["classification", "height", "geoid", "d_m"]
            )
        except Exception as e:
            print(f"Failed: {grain_id} — {e}")
            continue

        result = compute_metrics_old(df)

        if result is not None:
            result["grain_id"] = grain_id
            records.append(result)

print(f"\nComputed metrics for {len(records)} grains")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
out_csv = output_dir / f"wse_metrics_old.csv"
pd.DataFrame(records).to_csv(out_csv, index=False)

print(f"Saved to {out_csv}")

Found 54 chunk folders
Skipping chunk_10104_10554 (still running)
Skipping chunk_10554_11004 (still running)
Skipping chunk_11004_11454 (still running)
Processing chunk_1104_1554
Processing chunk_12354_12804
Processing chunk_12804_13254
Processing chunk_13254_13704
Processing chunk_13704_14154
Processing chunk_14154_14604
Processing chunk_14604_15054
Processing chunk_15054_15504
Skipping chunk_15504_15954 (still running)
Processing chunk_1554_2004
Skipping chunk_15954_16404 (still running)
Processing chunk_16404_16854
Processing chunk_16854_17304
Processing chunk_17304_17754
Processing chunk_17754_18204
Processing chunk_18204_18654
Processing chunk_18654_19104
Processing chunk_19104_19554
Processing chunk_19554_20004
Processing chunk_20004_20454
Processing chunk_2004_2454
Processing chunk_20454_20904
Processing chunk_204_654
Processing chunk_20904_21354
Processing chunk_21354_21804
Processing chunk_21804_22254
Processing chunk_22254_22704
Processing chunk_22704_23154
Processing chunk_2

In [1]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_chunk(chunk_dir):
    records = []

    flag_path = chunk_dir / "extraction_complete.flag"
    if not flag_path.exists():
        print(f"Skipping {chunk_dir.name} (still running)")
        return records

    print(f"Processing {chunk_dir.name}")

    parquet_files = sorted(chunk_dir.glob("*.parquet"))

    for parquet_file in parquet_files:
        grain_id = parquet_file.stem

        try:
            df = pd.read_parquet(
                parquet_file,
                columns=["classification", "height", "geoid", "d_m"]
            )
        except Exception as e:
            print(f"Failed: {grain_id} — {e}")
            continue

        result = compute_metrics_old(df)

        if result is not None:
            result["grain_id"] = grain_id
            records.append(result)

    return records

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import theilslopes

def robust_sigma(residuals):
    med = np.median(residuals)
    mad = np.median(np.abs(residuals - med))
    return 1.4826 * mad

def compute_metrics_old(df, BIN_WIDTH=50, MIN_BINS=5):
    
    # ---------------------------------
    # FILTER: classification == 4 only
    # ---------------------------------
    df = df[df["classification"] == 4].copy()
    
    if df.empty:
        return None

    # ---------------------------------
    # Compute WSE
    # ---------------------------------
    df["wse"] = df["height"] - df["geoid"]
    df = df[np.isfinite(df["wse"])]
    
    if df.empty:
        return None

    # ---------------------------------
    # Bin by distance
    # ---------------------------------
    df["bin"] = (df["d_m"] // BIN_WIDTH).astype(int)
    grouped = df.groupby("bin")

    bin_centers = []
    medians = []

    for b, g in grouped:
        bin_centers.append((b + 0.5) * BIN_WIDTH)
        medians.append(np.median(g["wse"]))

    if len(bin_centers) < MIN_BINS:
        return None

    d_vals = np.array(bin_centers)
    wse_vals = np.array(medians)

    # ---------------------------------
    # Theil-Sen slope
    # ---------------------------------
    slope, intercept, _, _ = theilslopes(wse_vals, d_vals)

    residuals = wse_vals - (intercept + slope * d_vals)
    sigma = robust_sigma(residuals)

    canal_length = d_vals.max() - d_vals.min()

    # ---------------------------------
    # Coverage + Continuity
    # ---------------------------------
    bins_sorted = np.sort(df["bin"].unique())

    min_bin = bins_sorted.min()
    max_bin = bins_sorted.max()
    total_possible_bins = max_bin - min_bin + 1

    coverage_frac = len(bins_sorted) / total_possible_bins

    # Longest contiguous run
    max_run = 1
    current_run = 1

    for i in range(1, len(bins_sorted)):
        if bins_sorted[i] == bins_sorted[i - 1] + 1:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1

    contig_frac = max_run / total_possible_bins

    # ---------------------------------
    # Signal-to-noise
    # ---------------------------------
    abs_slope = abs(slope)
    signal_amp = abs_slope * canal_length

    slope_snr = signal_amp / sigma if sigma > 0 else np.nan

    return {
        "n_bins": len(d_vals),
        "slope_m_per_m": slope,
        "abs_slope_m_per_m": abs_slope,
        "residual_sigma_robust_m": sigma,
        "canal_length_m": canal_length,
        "coverage_frac": coverage_frac,
        "contig_frac": contig_frac,
        "slope_snr": slope_snr,
    }

from pathlib import Path
import pandas as pd

input_dir = Path("/water1/mriduls/SOGRAIN_Analysis/us-south_from_204_to_25711/extracted_points")
output_dir = Path("/water1/mriduls/SOGRAIN_Analysis/us-south_from_204_to_25711/wse_metrics_old")
output_dir.mkdir(exist_ok=True)

records = []

# ------------------------------------------------------------
# Loop over chunk folders only
# ------------------------------------------------------------
chunk_dirs = sorted([d for d in input_dir.glob("chunk_*") if d.is_dir()])

print(f"Found {len(chunk_dirs)} chunk folders")



chunk_dirs = sorted([d for d in input_dir.glob("chunk_*") if d.is_dir()])

print(f"Found {len(chunk_dirs)} chunk folders")

all_records = []

N_WORKERS = 16  # adjust to CPU cores

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = [executor.submit(process_chunk, d) for d in chunk_dirs]

    for future in as_completed(futures):
        chunk_records = future.result()
        all_records.extend(chunk_records)

print(f"\nComputed metrics for {len(all_records)} grains")
out_csv = output_dir / f"wse_metrics_old.csv"
pd.DataFrame(all_records).to_csv(out_csv, index=False)

Found 57 chunk folders
Found 57 chunk folders
Processing chunk_10104_10554Processing chunk_11004_11454Processing chunk_10554_11004Processing chunk_1104_1554Processing chunk_11454_11904Processing chunk_11904_12354Processing chunk_12354_12804Processing chunk_12804_13254Processing chunk_13254_13704Processing chunk_14154_14604Processing chunk_13704_14154Processing chunk_14604_15054Processing chunk_15054_15504
Processing chunk_15504_15954Processing chunk_15954_16404Processing chunk_1554_2004














Processing chunk_16404_16854
Processing chunk_16854_17304
Processing chunk_17304_17754
Processing chunk_17754_18204
Processing chunk_18204_18654
Processing chunk_18654_19104
Processing chunk_19104_19554
Processing chunk_19554_20004
Processing chunk_20004_20454
Processing chunk_2004_2454
Processing chunk_20454_20904
Processing chunk_204_654
Processing chunk_20904_21354
Processing chunk_21354_21804
Processing chunk_21804_22254
Processing chunk_22254_22704
Processing chunk_22704_23154
Process